<h2 align="center">Machine Learning Model Development</h2>

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    BaggingRegressor,
    VotingRegressor
)
from sklearn.svm import SVR
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from sklearn.model_selection import GridSearchCV
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

In [ ]:
df = pd.read_excel("../../data/processed/one_year_mine_data_cleaned.xlsx")
df.head()

,Date,PM10,PM2.5,Temperature,Relative_humidity,RF-CUMULATIVE,Wind_speed,LOCATION,SEASON
0,2025-03-04,70.8,24.1,36.1,20.4,0.0,8.0,Near Office,Summer
1,2025-03-08,69.5,32.6,36.0,22.4,0.0,8.0,Near Office,Summer
2,2025-03-11,65.2,30.4,35.5,21.8,0.0,8.0,Near Office,Summer
3,2025-03-15,76.8,26.5,34.8,38.5,0.0,6.0,Near Office,Summer
4,2025-03-18,58.4,20.4,35.1,31.4,0.0,7.0,Near Office,Summer


In [3]:
features = [
    "Temperature",
    "Relative_humidity",
    "RF-CUMULATIVE",
    "Wind_speed"
]

target_pm10 = "PM10"
target_pm25 = "PM2.5"

In [4]:
X = df[features]
y = df[target_pm10]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

<h2 align="center">Linear Regression Model</h2>

In [5]:
lr = LinearRegression()
lr.fit(X_train, y_train)
pred = lr.predict(X_test)

In [6]:
mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)

print(mae)
print(rmse)
print(r2)

7.883535136113798
9.64471741172997
-0.013529600260708152


<h2 align="center">Decision Tree Model</h2>

In [7]:
# Decision Tree
dt = DecisionTreeRegressor(random_state=42)
dt.fit(X_train, y_train)
dt_pred = dt.predict(X_test)

In [8]:
mae = mean_absolute_error(y_test, dt_pred)
rmse = np.sqrt(mean_squared_error(y_test, dt_pred))
r2 = r2_score(y_test, dt_pred)

print(mae)
print(rmse)
print(r2)

8.570804597701148
10.498412099417482
-0.2008939438782611


<h2 align="center">Random Forest Model</h2>

In [9]:
# Random Forest
rf = RandomForestRegressor(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

In [10]:
mae = mean_absolute_error(y_test, rf_pred)
rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
r2 = r2_score(y_test, rf_pred)

print(mae)
print(rmse)
print(r2)

8.599286138669756
10.481828621725143
-0.19710303365317094


<h2 align="center">Bagging Model</h2>

In [11]:
# Bagging
try:
    bagging = BaggingRegressor(
        estimator=DecisionTreeRegressor(random_state=42),
        n_estimators=100,
        random_state=42
    )
except TypeError:
    bagging = BaggingRegressor(
        base_estimator=DecisionTreeRegressor(random_state=42),
        n_estimators=100,
        random_state=42
    )

bagging.fit(X_train, y_train)
bagging_pred = bagging.predict(X_test)

In [12]:
mae = mean_absolute_error(y_test, bagging_pred)
rmse = np.sqrt(mean_squared_error(y_test, bagging_pred))
r2 = r2_score(y_test, bagging_pred)

print(mae)
print(rmse)
print(r2)

8.533809180929225
10.420315277591298
-0.183093696312977


<h2 align="center">SVR Model</h2>

In [13]:
# SVR
svr = SVR(kernel="rbf")
svr.fit(X_train, y_train)
svr_pred = svr.predict(X_test)

In [14]:
mae = mean_absolute_error(y_test, svr_pred)
rmse = np.sqrt(mean_squared_error(y_test, svr_pred))
r2 = r2_score(y_test, svr_pred)

print(mae)
print(rmse)
print(r2)

7.705953182107911
9.577867223213
0.00047181189584355465


<h2 align="center">XGBoost Model</h2>

In [15]:
# XGBoost
xgb_model = xgb.XGBRegressor(
    objective="reg:squarederror",
    n_estimators=200,
    learning_rate=0.1,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)

In [16]:
mae = mean_absolute_error(y_test, xgb_pred)
rmse = np.sqrt(mean_squared_error(y_test, xgb_pred))
r2 = r2_score(y_test, xgb_pred)

print(mae)
print(rmse)
print(r2)

8.517835248749831
10.411093109499237
-0.18100050420377656


In [17]:
# Compare model performance in a table

results = []

models = {
    "Linear Regression": pred,
    "Decision Tree": dt_pred,
    "Random Forest": rf_pred,
    "Bagging": bagging_pred,
    "SVR": svr_pred,
    "XGBoost": xgb_pred,
}

for model_name, y_pred in models.items():
    results.append({
        "Model": model_name,
        "MAE": mean_absolute_error(y_test, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
        "R²": r2_score(y_test, y_pred)
    })

# Voting Regressor
try:
    voting = VotingRegressor([
        ("lr", LinearRegression()),
        ("dt", DecisionTreeRegressor(random_state=42)),
        ("rf", RandomForestRegressor(n_estimators=200, random_state=42)),
        ("xgb", xgb.XGBRegressor(
            objective="reg:squarederror",
            n_estimators=200,
            learning_rate=0.1,
            max_depth=4,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42
        ))
    ])
    voting.fit(X_train, y_train)
    voting_pred = voting.predict(X_test)

    results.append({
        "Model": "Voting",
        "MAE": mean_absolute_error(y_test, voting_pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, voting_pred)),
        "R²": r2_score(y_test, voting_pred)
    })
except Exception as e:
    print("Voting model could not be evaluated:", e)

results_df = pd.DataFrame(results)
results_df

,Model,MAE,RMSE,R²
0,Linear Regression,7.883535,9.644717,-0.013530
1,Decision Tree,8.570805,10.498412,-0.200894
2,Random Forest,8.599286,10.481829,-0.197103
3,Bagging,8.533809,10.420315,-0.183094
4,SVR,7.705953,9.577867,0.000472
5,XGBoost,8.517835,10.411093,-0.181001
6,Voting,8.275999,10.062630,-0.103266


In [18]:
Path("../data/processed").mkdir(parents=True, exist_ok=True)
results_df.to_excel(
    "../data/processed/results_data2.xlsx",
    index=False
)